In [ ]:
# Packages
from pathlib import Path

from resplotlib import rpc

# Directories
dir_path_repo = next(p for p in Path.cwd().resolve().parents if p.name == "resplotlib")
dir_path_data = dir_path_repo / "data"

# Read data
gdf_lines = gpd.read_file(dir_path_data / "transects.geojson").iloc[::100]


In [43]:
import ipywidgets as widgets
from ipyleaflet import Popup
from shapely.geometry import shape

output = widgets.Output()


# Define a function to handle the on-click event
def popup_handler(event, feature, **kwargs):
    # Get location
    location = shape(feature["geometry"]).centroid.coords[0][::-1]
    # with output:
    #     print(event)
    #     print(feature)
    #     print(kwargs)

    # Child
    child = widgets.HTML(f"transect_id: {feature['properties']['transect_id']}")

    # Create a popup with the content from GeoJSON properties
    popup = Popup(location=location, child=child, close_button=False, auto_close=True, name="popup")
    # Add the popup to the map
    m.add_layer(popup)


m = rpc.Map(bounds=gdf_lines.to_crs("EPSG:4326").total_bounds)
m = rpc.explore_geometries(
    gdf_lines,
    m=m,
    column="transect_id",
    legend=True,
    style_kwargs={"weight": 10},
    hover_style_kwargs={"weight": 15},
    name="transects",
)
m.layers[-1].on_click(popup_handler)


# Display the map
widgets.VBox([m, output])
